# EMA6938 - Data Science for Materials
## Week 8 Lab Notebook: Classification Methods

**Name:** *(your name here)*  
**Date:** *(date)*  
**Kernel:** Python (matds)

---

**Chapters:** Sandfeld Ch. 14  
**Format:** Take-home - due **Sunday 11:59 PM**  
**Dataset:** `data/week8_mp_classification.csv` (instructor-provided)

---

### How to use this notebook
- **Demo cells** (`# LECTURE DEMO`) reproduce examples from the lecture. Run them, understand them.
- **Task cells** (`# YOUR CODE HERE`) require you to write code.
- **Reflection cells** require written markdown answers. Replace the italic placeholder text.

This notebook has 7 parts:

| Part | Title | Connects to |
|------|-------|-------------|
| A | Load & Inspect | Lecture Segment 1 |
| B | Split & Scale | Lecture Segment 1 |
| C | Train Five Classifiers | Lecture Segment 2 |
| D | ROC and PR Curves | Lecture Segment 3 |
| E | Class Imbalance Handling | Lecture Segment 5 |
| F | Logistic Regression Coefficients | Lecture Segment 3 |
| G | Reflection | All segments |

---

**Submission:** Upload this `.ipynb` file to Canvas. Run `Kernel → Restart & Run All` before submitting to confirm all cells execute cleanly.

> **AI tool disclosure:** If you used any AI coding assistant (GitHub Copilot, ChatGPT, etc.) while completing this notebook, describe briefly which tool, for what purpose, and what you verified yourself. Delete this line if no AI tools were used.

In [ ]:
# Cell 0 - Environment check
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay, f1_score, precision_score, recall_score
)

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
print("✓ All imports successful")

---
## Part A - Load & Inspect

### A1: Load the dataset

In [ ]:
# Cell A1
# LECTURE DEMO
df = pd.read_csv('data/week8_mp_classification.csv')

print(f"Shape: {df.shape}")
print(f"\nColumn names (first 10): {df.columns[:10].tolist()}")
print(f"\nColumn names (last 10): {df.columns[-10:].tolist()}")
print(f"\n{df[['formula','band_gap','is_metallic','crystal_system']].head(8)}")

### A2: Class balance check

> **Always check class balance before splitting or training anything.**

In [ ]:
# Cell A2 - Value counts for the target variable
# TASK CELL
print("Class distribution:")
print(df['is_metallic'].value_counts())
print(f"\nClass proportions:")
print(df['is_metallic'].value_counts(normalize=True).round(3))

# Bar chart
fig, ax = plt.subplots(figsize=(5, 3))
# YOUR CODE: plot value counts for is_metallic as a bar chart
# Label axes clearly. Title: 'Class Balance - Metallic vs. Insulating'

plt.tight_layout()
plt.savefig('A2_class_balance.png', dpi=150)
plt.show()

print(f"\nMetallic-to-insulating ratio: {df['is_metallic'].mean()/(1-df['is_metallic'].mean()):.2f}:1")

### A3: Feature matrix setup

In [ ]:
# Cell A3 — Identify MAGPIE feature columns (all float columns except known non-feature columns)
# LECTURE DEMO
non_feature_cols = ['mp_id', 'formula', 'band_gap', 'Ef_eV_atom', 'volume_A3',
                    'density_g_cm3', 'crystal_system', 'is_metallic', 'is_insulating',
                    'composition']  # composition object column from featurization

feature_cols = [c for c in df.columns if c not in non_feature_cols
                and df[c].dtype in ['float64', 'float32']]

print(f"Number of MAGPIE features: {len(feature_cols)}")

# Set up X and y
X = df[feature_cols].values
y = df['is_metallic'].values

# Drop rows with NaN in X or y
mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
X = X[mask]
y = y[mask]

print(f"Final X shape: {X.shape}")
print(f"Final y shape: {y.shape}")
print(f"Class balance after cleaning: {y.mean():.1%} metallic")

---
## Part B - Split & Scale

### B1: Stratified train/test split

> **Rule:** stratify=y ensures class balance is preserved in both splits.

In [ ]:
# Cell B1
# LECTURE DEMO
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Train class balance: {y_train.mean():.1%} metallic")
print(f"Test class balance:  {y_test.mean():.1%} metallic")

### B2: StandardScaler - fit on training set ONLY

In [ ]:
# Cell B2
# LECTURE DEMO
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)          # transform only on test (no fit!)

print(f"X_train_scaled mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"X_train_scaled std  (should be ~1): {X_train_scaled.std():.4f}")
print(f"X_test_scaled  mean (not guaranteed ~0): {X_test_scaled.mean():.4f}")

---
## Part C - Train Five Classifiers

### C1: Logistic Regression

In [ ]:
# Cell C1
# LECTURE DEMO
lr = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=['insulating', 'metallic']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.3f}")

### C2: k-Nearest Neighbours

In [ ]:
# Cell C2
# LECTURE DEMO
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
y_prob_knn = knn.predict_proba(X_test_scaled)[:, 1]

print("=== k-Nearest Neighbours (k=5) ===")
print(classification_report(y_test, y_pred_knn, target_names=['insulating', 'metallic']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_knn):.3f}")

### C3: Gaussian Naive Bayes

In [ ]:
# Cell C3
# LECTURE DEMO
nb_clf = GaussianNB()
nb_clf.fit(X_train, y_train)  # NB does not need scaling

y_pred_nb = nb_clf.predict(X_test)
y_prob_nb = nb_clf.predict_proba(X_test)[:, 1]

print("=== Gaussian Naive Bayes ===")
print(classification_report(y_test, y_pred_nb, target_names=['insulating', 'metallic']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_nb):.3f}")

### C4: Decision Tree

In [ ]:
# Cell C4
# LECTURE DEMO
dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=SEED)
dt.fit(X_train_scaled, y_train)

y_pred_dt = dt.predict(X_test_scaled)
y_prob_dt = dt.predict_proba(X_test_scaled)[:, 1]

print("=== Decision Tree (max_depth=5) ===")
print(classification_report(y_test, y_pred_dt, target_names=['insulating', 'metallic']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_dt):.3f}")

### C5: Support Vector Machine

> **Note:** SVC with `kernel="rbf"` is slow on large datasets. If training takes >5 min, use the subsample below.

In [ ]:
# Cell C5 - Subsample if dataset is large
# LECTURE DEMO
MAX_SVC = 3000
if len(X_train_scaled) > MAX_SVC:
    idx = np.random.RandomState(SEED).choice(len(X_train_scaled), MAX_SVC, replace=False)
    X_svc = X_train_scaled[idx]
    y_svc = y_train[idx]
    print(f"Using subsample of {MAX_SVC} for SVC training")
else:
    X_svc, y_svc = X_train_scaled, y_train

svm = SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, random_state=SEED)
svm.fit(X_svc, y_svc)

y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

print("=== Support Vector Machine (RBF kernel) ===")
print(classification_report(y_test, y_pred_svm, target_names=['insulating', 'metallic']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_svm):.3f}")

### C6: Task - Comparison Table

Fill in the table below with your results. Then answer the reflection question.

In [ ]:
# Cell C6 — Helper function to compute all metrics
# LECTURE DEMO
def get_metrics(y_true, y_pred, y_prob, name):
    return {
        'Model': name,
        'Accuracy':  round((y_true == y_pred).mean(), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall':    round(recall_score(y_true, y_pred), 3),
        'F1':        round(f1_score(y_true, y_pred), 3),
        'ROC-AUC':   round(roc_auc_score(y_true, y_prob), 3),
    }

results = [
    get_metrics(y_test, y_pred_lr,  y_prob_lr,  'Logistic Regression'),
    get_metrics(y_test, y_pred_knn, y_prob_knn, 'k-NN (k=5)'),
    get_metrics(y_test, y_pred_nb,  y_prob_nb,  'Naive Bayes'),
    get_metrics(y_test, y_pred_dt,  y_prob_dt,  'Decision Tree'),
    get_metrics(y_test, y_pred_svm, y_prob_svm, 'SVM (RBF)'),
]

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.to_string())

**Reflection C6 - Fill in this cell:**

Which model achieved the highest ROC-AUC? Which achieved the best recall for the minority (insulating) class?
Describe the decision boundary of your best model geometrically. Is it a hyperplane, a curved surface, or a set of axis-aligned boxes?

*Your answer here*

---
## Part D - ROC and PR Curves

### D1: ROC curves for all five models

In [ ]:
# Cell D1
# LECTURE DEMO
fig, ax = plt.subplots(figsize=(7, 5))

for name, y_pred, y_prob in [
    ('Logistic Regression', y_pred_lr, y_prob_lr),
    ('k-NN',                y_pred_knn, y_prob_knn),
    ('Naive Bayes',         y_pred_nb, y_prob_nb),
    ('Decision Tree',       y_pred_dt, y_prob_dt),
    ('SVM (RBF)',           y_pred_svm, y_prob_svm),
]:
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name, ax=ax)

ax.plot([0,1],[0,1],'k--',label='Random (AUC=0.50)')
ax.set_title('ROC Curves — Metallic vs. Insulating Classification')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('D1_roc_curves.png', dpi=150)
plt.show()

### D2: Precision-Recall curve for the best model

In [ ]:
# Cell D2 — YOUR TASK: replace 'y_prob_lr' with the probability output of YOUR best model
# LECTURE DEMO
fig, ax = plt.subplots(figsize=(6, 4))
PrecisionRecallDisplay.from_predictions(y_test, y_prob_lr,   # <-- change to your best model
    name='Logistic Regression', ax=ax)
ax.set_title('Precision-Recall Curve — Best Model')
plt.tight_layout()
plt.savefig('D2_pr_curve.png', dpi=150)
plt.show()

### D3: Task - Threshold Tuning

> Find the decision threshold at which precision >= 0.85 for the metallic class. Report the recall at that threshold.

In [ ]:
# Cell D3
# TASK CELL
from sklearn.metrics import precision_recall_curve

precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_prob_lr)

# Find thresholds where precision >= 0.85
# YOUR CODE HERE
# Hint: iterate over (precision_vals, recall_vals, thresholds)
# and find the threshold where precision >= 0.85

# Print: threshold value, precision at that threshold, recall at that threshold

---
## Part E - Class Imbalance Handling

### E1: Logistic Regression WITHOUT class_weight (naive baseline)

In [ ]:
# Cell E1
# LECTURE DEMO
lr_naive = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
lr_naive.fit(X_train_scaled, y_train)

y_pred_naive = lr_naive.predict(X_test_scaled)
y_prob_naive = lr_naive.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression — NO class_weight ===")
print(classification_report(y_test, y_pred_naive, target_names=['insulating','metallic']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_naive):.3f}")

### E2: SMOTE - Synthetic Minority Oversampling

> Install if needed: `pip install imbalanced-learn`

In [ ]:
# Cell E2
# LECTURE DEMO
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

print(f"Original training class balance: {y_train.mean():.1%} metallic")
print(f"SMOTE training class balance:    {y_train_sm.mean():.1%} metallic")
print(f"SMOTE training size: {len(X_train_sm):,}")

lr_smote = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
lr_smote.fit(X_train_sm, y_train_sm)

y_pred_smote = lr_smote.predict(X_test_scaled)
y_prob_smote = lr_smote.predict_proba(X_test_scaled)[:, 1]

print("\n=== Logistic Regression — SMOTE ===")
print(classification_report(y_test, y_pred_smote, target_names=['insulating','metallic']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_smote):.3f}")

### E3: Task - Comparison Table + Reflection

In [ ]:
# Cell E3
imbalance_results = [
    get_metrics(y_test, y_pred_naive, y_prob_naive, 'LR — no handling'),
    get_metrics(y_test, y_pred_lr,    y_prob_lr,    'LR — class_weight'),
    get_metrics(y_test, y_pred_smote, y_prob_smote, 'LR — SMOTE'),
]
print(pd.DataFrame(imbalance_results).set_index('Model').to_string())

**Reflection E3 - Fill in this cell:**

Compare the three rows in the table. What effect did `class_weight="balanced"` have on precision vs. recall compared to the naive baseline?
What effect did SMOTE have?

For a high-throughput materials screening pipeline, where you are filtering 100,000 candidate compounds and will experimentally test everything that passes the filter, which approach would you use, and why?

*Your answer here*

---
## Part F - Logistic Regression Coefficients

### F1: Top features for metallic vs. insulating prediction

In [ ]:
# Cell F1
coef_series = pd.Series(lr.coef_[0], index=feature_cols)

print("Top 10 features predicting METALLIC (positive coefficients):")
print(coef_series.nlargest(10).round(4).to_string())

print("\nTop 10 features predicting INSULATING (negative coefficients):")
print(coef_series.nsmallest(10).round(4).to_string())

### F2: Coefficient bar chart

In [ ]:
# Cell F2
top10_pos = coef_series.nlargest(10)
top10_neg = coef_series.nsmallest(10)
top20 = pd.concat([top10_neg, top10_pos]).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#0D9488' if v > 0 else '#EF4444' for v in top20.values]
ax.barh(range(len(top20)), top20.values, color=colors)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f.replace('MagpieData ','') for f in top20.index], fontsize=9)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Logistic Regression Coefficient')
ax.set_title('Top Features for Metallic vs. Insulating Classification\n(green = predicts metallic, red = predicts insulating)')
plt.tight_layout()
plt.savefig('F2_lr_coefficients.png', dpi=150)
plt.show()

### F3: Task - Physical interpretation

Interpret the top positive and top negative coefficients physically.

- **Top positive feature (predicts metallic):** Why would high values of this composition descriptor suggest metallic character? Connect to band structure or bonding.
- **Top negative feature (predicts insulating):** Why would high values predict insulating character?
- **Compare to Week 5/6:** Do the same MAGPIE features that predicted high bandgap (Week 5) also predict insulating character here?

*Your answer here*

---
## Part G - Reflection

### G1: Connection to Week 5 RF Feature Importance

In 3-4 sentences, connect the logistic regression coefficients from Part F to the random forest feature importance from Week 5 (bandgap prediction).

- Do the same features appear near the top?
- If a feature predicted HIGH bandgap in Week 5 (positive RF importance), does it also predict INSULATING here (negative LR coefficient)?
- What does this pattern tell you about the relationship between the two models?

*Your answer here*

### G2: Classification problem in your own research

Think of a binary (or multi-class) classification problem relevant to your own research. In 3–4 sentences:

1. What are the target **classes**? (e.g. stable/unstable, conductor/insulator, crystalline/amorphous)
2. What **features** would you use?
3. Which **metric** would you optimise? Accuracy, precision, recall, F1, or AUC and why?

*Your answer here*

---
## Day 2 Live Demo

> **This section is covered during the Day 2 deep dive session.**

### Demo 1 - Decision boundary visualisation in 2D MAGPIE subspace

In [ ]:
# Cell Demo 1 — Visualise decision boundaries in 2D PCA subspace
# LECTURE DEMO

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Project to 2D for visualisation
pca2 = PCA(n_components=2, random_state=42)
X_tr_2d = pca2.fit_transform(X_train_scaled)
X_te_2d = pca2.transform(X_test_scaled)

# Fit LR and DT in 2D space for boundary visualisation
lr_2d = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)
dt_2d = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
lr_2d.fit(X_tr_2d, y_train); dt_2d.fit(X_tr_2d, y_train)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, model, title in zip(axes, [lr_2d, dt_2d],
                             ['Logistic Regression (linear boundary)',
                              'Decision Tree (axis-aligned boundary)']):
    xx, yy = np.meshgrid(np.linspace(X_tr_2d[:,0].min()-1, X_tr_2d[:,0].max()+1, 200),
                         np.linspace(X_tr_2d[:,1].min()-1, X_tr_2d[:,1].max()+1, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    ax.scatter(X_te_2d[:,0], X_te_2d[:,1], c=y_test,
               cmap='RdBu', alpha=0.5, s=10, edgecolors='none')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_title(title, fontsize=10)
plt.suptitle('Decision boundaries in PC1–PC2 space (test set coloured by true class)',
             fontsize=11)
plt.tight_layout()
plt.savefig('Day2_decision_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()

### Demo 2 - Threshold sensitivity: precision–recall trade-off

In [ ]:
# Cell Demo 2 — Threshold sensitivity for the best classifier
# LECTURE DEMO

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# Use the best model from Part C (assumed to be rf or lr — adjust as needed)
# Get probability scores for the positive class (metallic=1)
try:
    probs = rf.predict_proba(X_test_scaled)[:, 1]
    model_name = 'Random Forest'
except:
    probs = lr.predict_proba(X_test_scaled)[:, 1]
    model_name = 'Logistic Regression'

thresholds = np.linspace(0.1, 0.9, 50)
precisions, recalls, f1s = [], [], []

for thresh in thresholds:
    y_pred_t = (probs >= thresh).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precisions, color='#EF4444', lw=2, label='Precision')
ax.plot(thresholds, recalls,   color='#0D9488', lw=2, label='Recall')
ax.plot(thresholds, f1s,       color='#7C3AED', lw=2, label='F1', ls='--')
ax.axvline(0.5, color='black', lw=1, ls=':', label='Default threshold (0.5)')
ax.set_xlabel('Decision threshold'); ax.set_ylabel('Score')
ax.set_title(f'{model_name} — precision/recall/F1 vs. threshold')
ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('Day2_threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

best_f1_idx = np.argmax(f1s)
print(f"Best F1 = {f1s[best_f1_idx]:.3f} at threshold = {thresholds[best_f1_idx]:.2f}")
print(f"  Precision at best F1: {precisions[best_f1_idx]:.3f}")
print(f"  Recall    at best F1: {recalls[best_f1_idx]:.3f}")

### Demo 3 - Logistic regression coefficients: composition chemistry of metallicity

In [ ]:
# Cell Day2-3 — LR coefficients: which features predict metallic vs. insulating?
# LECTURE DEMO

import pandas as pd
import matplotlib.pyplot as plt

coef_series = pd.Series(lr.coef_[0], index=feature_cols).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
coef_series.head(10).plot(kind='barh', color='#0D9488', ax=axes[0])
axes[0].set_title('Top 10 — predicts INSULATING (negative coef)', fontsize=10)
axes[0].set_xlabel('Coefficient value')

coef_series.tail(10).plot(kind='barh', color='#EF4444', ax=axes[1])
axes[1].set_title('Top 10 — predicts METALLIC (positive coef)', fontsize=10)
axes[1].set_xlabel('Coefficient value')

plt.suptitle('Logistic Regression coefficients — metallic vs. insulating prediction',
             fontsize=11)
plt.tight_layout()
plt.savefig('Day2_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print("Features most predictive of INSULATING character (top 5 negative):")
print(coef_series.head(5).round(4).to_string())
print()
print("Features most predictive of METALLIC character (top 5 positive):")
print(coef_series.tail(5).round(4).to_string())

**Day 2 Discussion questions:**

1. From Demo 1: how does the logistic regression boundary (linear hyperplane) differ from the decision tree boundary (axis-aligned steps) in PC space? Which better matches the physical boundary between metallic and insulating oxides?

2. From Demo 2: if you were building a high-throughput screening pipeline to find insulating oxides, would you raise or lower the threshold from 0.5? What do you sacrifice in exchange?

3. From Demo 3: compare the LR coefficients to the RF feature importances from Part C. Do the same features appear at the top? If they disagree, what does that tell you about the different ways the two models learn the metallic/insulating boundary?

---
## Submission Checklist

Before submitting, confirm all cells have been executed:

- [ ] A2: Class balance bar chart saved
- [ ] C6: Comparison table printed and reflection filled in
- [ ] D1: ROC curves for all five models saved
- [ ] D2: PR curve for best model saved
- [ ] D3: Threshold tuning answer filled in
- [ ] E3: Imbalance comparison table and reflection filled in
- [ ] F2: Coefficient bar chart saved
- [ ] F3, G1, G2, Day2: All reflection cells answered
- [ ] All reflection cells answered (no placeholder text)
- [ ] AI disclosure note updated or deleted at the top of the notebook
- [ ] File renamed: `[LastName]_week8.ipynb`
**Final check:** Run `Kernel → Restart & Run All`. All cells must execute without errors before submitting.